# Project 3 - Tips Patterns in NYC


- **Dataset(s) to be used:** 
[2022 Yellow Taxi Trip Data](https://data.cityofnewyork.us/Transportation/2022-Yellow-Taxi-Trip-Data/qp3b-zxtp/about_data)
, [U.S. Federal Holidays 2022](https://www.kaggle.com/datasets/donnetew/us-holiday-dates-2004-2021)

- **Analysis question:** How do tipping behaviors differ across **night vs daytime**, **weekend vs weekday**, and **holiday vs non-holiday** taxi trips in New York City during 2022?

From taxi dataset:
- `tpep_pickup_datetime` (for extracting date & hour)
- `tip_amount`
- `fare_amount`
- `payment_type` (to filter only card payments)
- optional for filtering quality: `trip_distance`, `passenger_count`
From holiday dataset:
- `date` (to merge)
- `holiday_name`


**Hypothesis**:

Taxi riders will tip a **higher percentage**:
- at **night** (8pm–4am)
- on **weekends**
- and **on holidays**

compared to daytime, weekdays, and non-holiday trips.

### Data Preparation 
1. Filter to **2022** only
2. Filter to **credit-card payment only**
3. Compute **tip percentage**:  
   `tip_percentage = (tip_amount / fare_amount) * 100`
4. Create categorical flags:
   - **Night** = 20:00–03:59
   - Weekend vs Weekday
   - Holiday vs Non-Holiday
5. Merge taxi data with holiday dataset on pickup date
6. Group & compare mean tip percentage across categories
7. Visualize distributions and regression trends


In [ ]:
import pandas as pd

df = pd.read_csv("2022_Yellow_Taxi_Trip_Data_20251126.csv", low_memory=False)
df.head()

ModuleNotFoundError: No module named 'pandas'

Now we would like to explore the payment types to clean the data properly 

In [ ]:
# explore payment types + tips BEFORE CLEANING

# ensure numeric columns
df["payment_type"] = pd.to_numeric(df["payment_type"], errors="coerce")
df["tip_amount"] = pd.to_numeric(df["tip_amount"], errors="coerce")
df["fare_amount"] = pd.to_numeric(df["fare_amount"], errors="coerce")

# groupby exploration
print("Payment Type Counts:")
print(df["payment_type"].value_counts())

print("\nAverage Tip Amount by Payment Type:")
print(df.groupby("payment_type")["tip_amount"].mean())

print("\nPercent of Zero Tips by Payment Type:")
print(
    df.groupby("payment_type")["tip_amount"]
    .apply(lambda x: (x == 0).mean() * 100)
    .round(2)
)


Payment Type Counts:
payment_type
1    30085763
2     7763339
0     1368303
4      244364
3      194323
5           6
Name: count, dtype: int64

Average Tip Amount by Payment Type:
payment_type
0    3.759275
1    3.452928
2    0.001077
3    0.001862
4    0.050724
5    0.000000
Name: tip_amount, dtype: float64

Percent of Zero Tips by Payment Type:
payment_type
0     13.41
1      3.99
2     99.98
3     97.89
4     98.57
5    100.00
Name: tip_amount, dtype: float64


In [ ]:
df[df["payment_type"] == 0].head()


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
2392428,2,01/01/2022 12:50:00 AM,01/01/2022 12:54:00 AM,NaN,1,NaN,NaN,68,246,0,13.20,0.0,0.5,1.75,0.0,0.3,18.25,NaN,NaN
2392429,2,01/01/2022 12:49:24 AM,01/01/2022 01:27:36 AM,NaN,13.31,NaN,NaN,257,223,0,44.87,0.0,0.5,10.05,0.0,0.3,55.72,NaN,NaN
2392430,2,01/01/2022 12:42:00 AM,01/01/2022 12:56:00 AM,NaN,2.87,NaN,NaN,143,236,0,13.23,0.0,0.5,3.51,0.0,0.3,20.04,NaN,NaN
2392431,2,01/01/2022 12:40:00 AM,01/01/2022 12:55:00 AM,NaN,3.24,NaN,NaN,143,262,0,14.19,0.0,0.5,3.72,0.0,0.3,21.21,NaN,NaN
2392432,2,01/01/2022 12:40:00 AM,01/01/2022 12:52:00 AM,NaN,2.19,NaN,NaN,239,166,0,13.20,0.0,0.5,5.25,0.0,0.3,21.75,NaN,NaN


In [ ]:
#Filter to credit card trips only because other payments are 
df = df[df["payment_type"] == 1]

#Keep only relevant columns to cut memory
df = df[[
    "tpep_pickup_datetime",
    "fare_amount",
    "tip_amount",
    "trip_distance"
]]

#Remove rows with invalid fares or missing tips
df = df[(df["fare_amount"] > 0) & (df["tip_amount"] >= 0)]

#Convert datetime
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")

#Remove rows where datetime failed to parse
df.dropna(subset=["tpep_pickup_datetime"], inplace=True)

#Report memory usage after filtering
df.memory_usage(deep=True).sum() / 1e9


> Note: I filter to credit-card payments because cash trips do not record tip_amount in the dataset, which results in many false zeros and would bias the analysis toward lower tip percentages.


In [ ]:
df = df[df["payment_type"] == 1]


Before analyzing tips, we need to convert the tip amount into a tip percentage. Looking only at the dollar value of a tip can be misleading because the generosity of a tip depends on the cost of the trip. A $5 tip on a long and expensive ride is not as generous as a $1 tip on a short and cheap ride. By calculating the percentage of the tip relative to the fare amount, we can compare tipping behavior more fairly across different trip lengths and prices. This allows us to better understand how tipping changes during different times of day, on weekends versus weekdays, and on holidays versus non-holidays. Since the project focuses on understanding what affects tipping behavior, tip percentage is the most meaningful variable for our analysis.

In [ ]:
# Remove negative or zero fare amounts and invalid/negative tips
df = df[(df["fare_amount"] > 0) & (df["tip_amount"] >= 0)]

# Convert pickup datetime
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")

# Drop rows where datetime could not be parsed
df.dropna(subset=["tpep_pickup_datetime"], inplace=True)

# Compute tip percentage
df["tip_percentage"] = (df["tip_amount"] / df["fare_amount"]) * 100

# Report memory usage after filtering & conversions
df.memory_usage(deep=True).sum() / 1e9


/var/folders/k1/lny40b654c35nk73h3xmgr1h0000gn/T/ipykernel_72613/3246920259.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")


np.float64(10.925541946)

Now I will convert large columns (IDs, passenger_count, fare/tip amounts, etc.) into smaller types with no loss of precision to lower the memory usage 

In [ ]:
# Only convert the numeric columns I need
float_cols = ["trip_distance", "fare_amount", "tip_amount", "tip_percentage"]

for col in float_cols:
    # Remove thousands separators like "53,440.55" -> "53440.55"
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce", downcast="float")

df.memory_usage(deep=True).sum() / 1e9


np.float64(8.139206897)

In [ ]:
# Keep only the columns we actually need from here on
keep_cols = [
    "tpep_pickup_datetime",
    "fare_amount",
    "tip_amount",
    "tip_percentage",
    "trip_distance"
]

df = df[keep_cols].copy()

# Progress / sanity check
print("Columns now:", df.columns.tolist())
print("Rows:", len(df))

# Memory after dropping unused columns
df.memory_usage(deep=True).sum() / 1e9


Columns now: ['tpep_pickup_datetime', 'fare_amount', 'tip_amount', 'tip_percentage', 'trip_distance']
Rows: 30081453


np.float64(1.20325812)

In [ ]:
# Create time-based features

# Date at day-level (kept as datetime for later merging)
df["pickup_date"] = df["tpep_pickup_datetime"].dt.floor("D")

# Hour of the day (0–23)
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour

# Day of week (Monday=0, Sunday=6)
df["day_of_week"] = df["tpep_pickup_datetime"].dt.dayofweek

# Weekend flag: 1 if Saturday or Sunday, else 0
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Night flag: 1 if 8pm–3:59am, else 0
df["is_night"] = df["pickup_hour"].apply(lambda h: 1 if (h >= 20 or h < 4) else 0)

# Quick sanity check
df[["pickup_date", "pickup_hour", "day_of_week", "is_weekend", "is_night"]].head()


,pickup_date,pickup_hour,day_of_week,is_weekend,is_night
0,2022-01-01,0,5,1,1
1,2022-01-01,0,5,1,1
2,2022-01-01,0,5,1,1
4,2022-01-01,0,5,1,1
5,2022-01-01,0,5,1,1


In [ ]:
# Add readable day name (e.g., Monday, Saturday)
df["day_name"] = df["tpep_pickup_datetime"].dt.day_name()

# Quick check
df[["pickup_date", "day_of_week", "day_name", "is_weekend"]].head()


,pickup_date,day_of_week,day_name,is_weekend
0,2022-01-01,5,Saturday,1
1,2022-01-01,5,Saturday,1
2,2022-01-01,5,Saturday,1
4,2022-01-01,5,Saturday,1
5,2022-01-01,5,Saturday,1


In [ ]:
# Load U.S. holidays data
holidays = pd.read_csv("US Holiday Dates (2004-2021).csv")

# Convert to datetime
holidays["Date"] = pd.to_datetime(holidays["Date"], errors="coerce")

# Month-day key (e.g. "01-01", "07-04")
holidays["month_day"] = holidays["Date"].dt.strftime("%m-%d")

# Keep unique month-day + holiday name
holiday_md = holidays[["month_day", "Holiday"]].drop_duplicates()

# Create same month-day key in taxi data
df["month_day"] = df["pickup_date"].dt.strftime("%m-%d")

# Merge on month-day instead of full date
df = df.merge(holiday_md, on="month_day", how="left")

# Holiday flag: 1 if matched, 0 otherwise
df["is_holiday"] = df["Holiday"].notna().astype(int)

# (Optional) drop helper column
df.drop(columns=["month_day"], inplace=True)

# Quick sanity check: look around known holidays
df[df["pickup_date"].between("2022-07-04", "2022-07-04")][
    ["pickup_date", "day_name", "is_weekend", "Holiday", "is_holiday"]
].head()


,pickup_date,day_name,is_weekend,Holiday,is_holiday
15896922,2022-07-04,Monday,0,4th of July,1
15896939,2022-07-04,Monday,0,4th of July,1
15898423,2022-07-04,Monday,0,4th of July,1
15898527,2022-07-04,Monday,0,4th of July,1
15898594,2022-07-04,Monday,0,4th of July,1


In [ ]:
# Hypothesis testing: Compare mean tip percentage

def compare_tips(flag, label):
    result = df.groupby(flag)["tip_percentage"].mean().reset_index()
    result.columns = [flag, f"avg_tip_percentage_{label}"]
    return result

night_comp = compare_tips("is_night", "night")
weekend_comp = compare_tips("is_weekend", "weekend")
holiday_comp = compare_tips("is_holiday", "holiday")

print("Night vs Day:")
print(night_comp, "\n")

print("Weekend vs Weekday:")
print(weekend_comp, "\n")

print("Holiday vs Non-holiday:")
print(holiday_comp)


Night vs Day:
   is_night  avg_tip_percentage_night
0         0                 28.579335
1         1                 28.864405 

Weekend vs Weekday:
   is_weekend  avg_tip_percentage_weekend
0           0                   28.570998
1           1                   28.891280 

Holiday vs Non-holiday:
   is_holiday  avg_tip_percentage_holiday
0           0                   28.719293
1           1                   28.447667


Based on a full year of NYC taxi trips during 2022, we find that tipping behavior does vary by timing:

• Riders tip a slightly higher percentage at night (28.86% vs 28.58%)

• Riders tip more on weekends than weekdays (28.89% vs 28.57%)

• Surprisingly, holidays show slightly lower tipping (28.45% vs 28.72%)

These differences are small, but statistically meaningful due to the very large sample size (over 3 million trips). The first two results support our hypothesis, while the third goes against it — tipping does not increase on holidays.

In [ ]:
import plotly.express as px
from IPython.display import HTML


def plot_compare(df, x_col, y_col, title):
    fig = px.bar(
        df,
        x=x_col,
        y=y_col,
        text=df[y_col].round(2).astype(str) + "%",
        title=title,
        color=x_col
    )
    fig.update_layout(
        yaxis_title="Average Tip Percentage (%)",
        xaxis_title="Category",
        template="plotly_white",
        showlegend=False
    )
    # Format hover tooltips
    fig.update_traces(textposition="outside")
    fig.show()


# Prepare clean category labels
night_comp["category"] = night_comp["is_night"].map({0: "Day", 1: "Night"})
weekend_comp["category"] = weekend_comp["is_weekend"].map({0: "Weekday", 1: "Weekend"})
holiday_comp["category"] = holiday_comp["is_holiday"].map({0: "Non-Holiday", 1: "Holiday"})

# Plot Night vs Day
plot_compare(
    night_comp,
    "category",
    "avg_tip_percentage_night",
    "Average Tip %: Night vs Day"
)

# Plot Weekend vs Weekday
plot_compare(
    weekend_comp,
    "category",
    "avg_tip_percentage_weekend",
    "Average Tip %: Weekend vs Weekday"
)

# Plot Holiday vs Non-Holiday
plot_compare(
    holiday_comp,
    "category",
    "avg_tip_percentage_holiday",
    "Average Tip %: Holiday vs Non-Holiday"
)


1️⃣ Night vs Day

The results suggest that riders tip a slightly higher percentage at night (28.86%) compared to daytime (28.58%). Although the difference (+0.28 percentage points) is small, our large sample size means this is likely a statistically significant effect. This supports the hypothesis that nighttime travel encourages higher tipping behavior.

2️⃣ Weekend vs Weekday

On weekends, riders tipped an average of 28.89% compared to 28.58% on weekdays — a difference of +0.31 percentage points. This aligns with our hypothesis that weekend travel, potentially associated with leisure activities and nightlife, leads to more generous tipping.

3️⃣ Holiday vs Non-Holiday

Contrary to expectations, holidays showed a slightly lower average tip percentage (28.45%) vs non-holidays (28.72%), a decrease of −0.27 percentage points. This result rejects our hypothesis that tipping would increase on holidays. One possible explanation is that holiday surcharges or higher fares may offset tipping generosity.

In [ ]:

# Group by pickup hour
hourly_tips = df.groupby("pickup_hour")["tip_percentage"].mean().reset_index()

# Round values
hourly_tips["tip_percentage"] = hourly_tips["tip_percentage"].round(2)

fig = px.line(
    hourly_tips,
    x="pickup_hour",
    y="tip_percentage",
    markers=True,
    title="Average Tip Percentage by Hour of Day",
)

fig.update_layout(
    xaxis_title="Hour of Day (0–23)",
    yaxis_title="Average Tip Percentage (%)",
    template="plotly_white",
    hovermode="x unified"
)

HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))


The huge spike around 5 AM looks suspicious

→ That’s a data anomaly, likely caused by very few rides at that hour, so a couple of big-tip rides massively shift the mean.

In [ ]:
hourly_tips = df.groupby("pickup_hour")["tip_percentage"].median().reset_index()
hourly_tips["tip_percentage"] = hourly_tips["tip_percentage"].round(2)

fig = px.line(
    hourly_tips,
    x="pickup_hour",
    y="tip_percentage",
    markers=True,
    title="Median Tip Percentage by Hour of Day (More Reliable)"
)

fig.update_layout(
    xaxis_title="Hour of Day (0–23)",
    yaxis_title="Median Tip Percentage (%)",
    template="plotly_white",
    hovermode="x unified"
)

HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))


🧠 What does this median chart tell us?

✔ Tip percentage increases during the evening, especially around 18:00–21:00

✔ Late night (after 21:00) remains slightly higher than morning hours

✔ Very early morning (4–6AM) dips, fewer generous drunks than we expected 

✔ Range is narrower (around 25.5% → 27%), meaning median smooths out outliers

In [ ]:
weekend_tips = df.groupby("is_weekend")["tip_percentage"].median().reset_index()
weekend_tips["label"] = weekend_tips["is_weekend"].map({0: "Weekday", 1: "Weekend"})

fig = px.bar(
    weekend_tips,
    x="label",
    y="tip_percentage",
    text="tip_percentage",
    title="Median Tip Percentage: Weekend vs Weekday",
    labels={"label": "", "tip_percentage": "Median Tip Percentage (%)"},
    template="plotly_white"
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))


In [ ]:
holiday_tips = df.groupby("is_holiday")["tip_percentage"].median().reset_index()
holiday_tips["label"] = holiday_tips["is_holiday"].map({0: "Non-Holiday", 1: "Holiday"})

fig = px.bar(
    holiday_tips,
    x="label",
    y="tip_percentage",
    text="tip_percentage",
    title="Median Tip Percentage: Holiday vs Non-Holiday",
    labels={"label": "", "tip_percentage": "Median Tip Percentage (%)"},
    template="plotly_white"
)

fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))


In [ ]:
import plotly.express as px

# Create pivot table for heatmap
heatmap_data = df.pivot_table(
    index="pickup_hour", 
    columns="is_weekend",
    values="tip_percentage",
    aggfunc="median"
).reset_index()

# Rename columns for readability
heatmap_data.columns = ["pickup_hour", "Weekday", "Weekend"]

# Convert to long format for Plotly
heatmap_data_melted = heatmap_data.melt(
    id_vars="pickup_hour",
    value_vars=["Weekday", "Weekend"],
    var_name="Day Type",
    value_name="Median Tip %"
)

# Plot heatmap
fig = px.density_heatmap(
    heatmap_data_melted,
    x="Day Type",
    y="pickup_hour",
    z="Median Tip %",
    color_continuous_scale="Viridis",
    title="Heatmap: Median Tip % by Hour & Day Type"
)

fig.update_layout(
    xaxis_title="Day Type",
    yaxis_title="Hour of Day (0–23)",
    template="plotly_white"
)

HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))


In [ ]:
# Create pivot table for median tip %
heatmap_data = df.pivot_table(
    index="pickup_hour",
    columns="is_weekend",
    values="tip_percentage",
    aggfunc="median"
).reset_index()

# Rename columns for readability
heatmap_data.columns = ["pickup_hour", "Weekday", "Weekend"]

# Melt for plotting
heatmap_melt = heatmap_data.melt(
    id_vars="pickup_hour",
    value_vars=["Weekday", "Weekend"],
    var_name="Day Type",
    value_name="Median Tip %"
)

# Plot
fig = px.density_heatmap(
    heatmap_melt,
    x="Day Type",
    y="pickup_hour",
    z="Median Tip %",
    color_continuous_scale="RdYlGn",
    title="Heatmap: Median Tip % by Hour & Day Type",
)

fig.update_layout(
    xaxis_title="Day Type",
    yaxis_title="Hour of Day (0–23)",
    yaxis=dict(autorange="reversed"),  # Show midnight on top
    template="plotly_white"
)

fig.update_coloraxes(colorbar_title="Median Tip %")


HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))


**Heatmap Interpretation**

The heatmap shows median tip percentage by hour of day and whether the ride
was on a weekday or weekend. Colors are very similar across the Weekend and
Weekday columns, confirming that the calendar day has almost no effect on
tipping behavior. There is a mild pattern by time of day: tips are slightly
higher in the evening and late night hours and slightly lower in the early
morning and mid-day. Overall, time of day matters more than whether it is a
weekend or a weekday.


**Conclusion**

Time of day influences tipping more than whether it is a weekend or holiday.
People tip a little more at night, but weekends and holidays do not significantly change tipping behavior.

Although the differences are small (less than 1 percentage point), the massive sample size makes them statistically real, just not financially meaningful.